# 3.1.3, Proof of the six tilde-side Cartan relations on V

**Goal.** Step by step within the system, sequentially close the six
operator relations that are the Koszul/tilde counterparts of the seven
Cartan relations on the standard side (3.1.2), on a generic
multivector $V$.

The tilde operators on a Poisson manifold $(M,\pi)$:
$\tilde{\iota}_\omega$ contracts a multivector against $\omega$ in
**one slot**; $\tilde{d}_\pi$ raises the degree by one via the
Schouten–Nijenhuis bracket
$\tilde{d}_\pi V := [\pi, V]_{\mathrm{SN}}$;
$\tilde{\mathcal{L}}_\omega := \tilde{d}\,\tilde{\iota}_\omega +
\tilde{\iota}_\omega\,\tilde{d}$ is the Lie derivative.

| # | Relation | Meaning | Poisson? |
|---|---|---|---|
| 1 | $\tilde{\iota}_\omega\,\tilde{\iota}_\eta + \tilde{\iota}_\eta\,\tilde{\iota}_\omega = 0$ | interior products anti-commute | no |
| 2 | $\tilde{d}^2 V = 0$ | nilpotency of the exterior derivative | **yes** |
| 3 | $\tilde{\mathcal{L}}_\omega V = \tilde{d}\,\tilde{\iota}_\omega V + \tilde{\iota}_\omega\,\tilde{d} V$ | Cartan's magic formula | no |
| 4 | $[\tilde{\mathcal{L}}_\alpha, \tilde{\iota}_\beta] = \tilde{\iota}_{[\alpha,\beta]_K}$ | Lie / interior commutator | no |
| 5 | $[\tilde{\mathcal{L}}_\alpha, \tilde{d}] V = 0$ | Lie commutes with $\tilde{d}$ | **yes** |
| 6 | $[\tilde{\mathcal{L}}_\alpha, \tilde{\mathcal{L}}_\beta] V = \tilde{\mathcal{L}}_{[\alpha,\beta]_K} V$ | commutator of Lie derivatives | **yes** |

Proof chains are rendered as LaTeX via `display_chain`.


## Strategy

Unlike the standard side, here we use a **single engine**:
`KoszulProblem.tilde_intrinsic_engine()` returns 28 rules. The engine
contains the rules added during Phase 14:

- **14.A–E**, intrinsic expansion for interior product, $\tilde{d}$,
  $\tilde{\mathcal{L}}$ (slot Leibniz, alternating, anchor
  anti-symmetry, etc.) + 6 closure axioms.
- **14.F**, the defining version of Cartan's magic formula
  (`TildeCartanMagicDefinition`).
- **14.G**, `WrappedPairingAnchorAntisymmetryDefinition` together
  with `TildeSnJacobiResidueDefinition`: the first rule cancels the
  paired bracket-ring residues from
  $\langle \pi^\sharp a, b \rangle + \langle \pi^\sharp b, a \rangle = 0$;
  the second recognizes the canonical 5-term form of the
  Schouten–Nijenhuis Jacobi obstruction
  $[\pi,\pi]_{\mathrm{SN}}(\alpha,\beta,\gamma) = 0$ (Poisson-gated).

The proof goes through one call: `prob.prove_tilde_cartan(LHS, RHS,
etas=(η_1, …, η_q))`. The system first opens a multi-evaluation in a
well-informed order, then runs the engine to a fixed point. If it
reduces to zero we get a `ProofChain`; otherwise a `ProofFailure`.

Relations **2/5/6** require the Poisson assumption: in those cells we
call `prob.assume_poisson()` first, which activates the Poisson-gated
rules.


In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401


## 1. Setup, symbols, problem, helper function

Symbols used below:

- $\pi$: a generic Poisson bivector; before relations 2/5/6 we open
  $[\pi,\pi]_{\mathrm{SN}} = 0$ via `prob.assume_poisson()`.
- $V$: a generic multivector; each relation receives it at its
  natural SN-degree (1-VF or 2-VF).
- $\omega, \eta, \alpha, \beta, \xi$: generic 1-forms
  (`Graded(degree=1)`).

For each relation we set up a separate `KoszulProblem` instance; the
helper `_build` (the exact counterpart of `_build_problem` in the
tests) returns the problem, registry, $\pi$, $V$ and the forms.


In [2]:
from jacopy.algebra.derivation import Act
from jacopy.calculus.tilde import (
    TildeExteriorDerivative,
    TildeInteriorProduct,
    TildeLieDerivative,
)
from jacopy.core.expr import Integer, Neg, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.display.jupyter import display_chain
from jacopy.library.koszul_problem import KoszulProblem


def _build(form_names=("ω", "η"), V_degree=1):
    """Same setup as ``_build_problem`` in the tests."""
    reg = PropertyRegistry()
    pi = Symbol("π")
    forms = tuple(Symbol(n) for n in form_names)
    for f in forms:
        reg.declare(f, Graded(degree=1))
    V = Symbol("V")
    reg.declare(V, Graded(degree=V_degree))
    prob = KoszulProblem(
        pi,
        forms,
        registry=reg,
        multivectors=((V, V_degree),),
    )
    return (prob, reg, pi, V) + forms


def prove(label, prob, lhs, rhs, etas):
    """Single-call interface; prints step count and returns a ProofChain."""
    chain = prob.prove_tilde_cartan(lhs, rhs, etas=etas)
    print(f"{label} -> closed in {len(chain)} steps")
    return chain


_probe, *_ = _build()
print("Engine rule count:", len(_probe.tilde_intrinsic_engine().definitions))


Engine rule count: 28


## 2. Relation 1: $\tilde{\iota}$ anti-commute

$$
\tilde{\iota}_\omega\,\tilde{\iota}_\eta\,V \;+\;
\tilde{\iota}_\eta\,\tilde{\iota}_\omega\,V \;=\; 0.
$$

**Shape.** Each $\tilde{\iota}$ consumes a form-slot; we pick $V$
as a 2-vector so the result still leaves an evaluable slot. Evaluating
against a single 1-form $\xi$, MultiEval's alternating-canonicalisation
reduces the two-term sum to zero:

$$
V(\omega, \eta, \xi) \;+\; V(\eta, \omega, \xi) \;=\; 0.
$$


In [3]:
prob, _, _, V, omega, eta = _build(V_degree=2)
xi = Symbol("ξ")
prob.registry.declare(xi, Graded(degree=1))

lhs = Sum.make(
    Act(TildeInteriorProduct(omega), Act(TildeInteriorProduct(eta), V)),
    Act(TildeInteriorProduct(eta), Act(TildeInteriorProduct(omega), V)),
)
chain = prove("(2-VF, eval ξ) ι̃_ω ι̃_η V + ι̃_η ι̃_ω V = 0",
              prob, lhs, Integer(0), etas=(xi,))
display_chain(chain)


(2-VF, eval ξ) ι̃_ω ι̃_η V + ι̃_η ι̃_ω V = 0 -> closed in 10 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(\tilde{\iota}_{\omega}\!\left(\tilde{\iota}_{\eta}\!\left(V\right)\right) + \tilde{\iota}_{\eta}\!\left(\tilde{\iota}_{\omega}\!\left(V\right)\right)\right)\!\left(\xi\right) \to \left(\tilde{\iota}_{\omega}\!\left(\tilde{\iota}_{\eta}\!\left(V\right)\right)\right)\!\left(\xi\right) + \left(\tilde{\iota}_{\eta}\!\left(\tilde{\iota}_{\omega}\!\left(V\right)\right)\right)\!\left(\xi\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(\tilde{\iota}_{\omega}\!\left(\tilde{\iota}_{\eta}\!\left(V\right)\right)\right)\!\left(\xi\right) \to \left(\tilde{\iota}_{\eta}\!\left(V\right)\right)\!\left(\omega,\, \xi\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{\iota}_{\eta}\!\left(V\right)\right)\!\left(\omega,\, \xi\right) \to V\!\left(\eta,\, \omega,\, \xi\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
V\!\left(\eta,\, \omega,\, \xi\right) \to -V\!\left(\eta,\, \xi,\, \omega\right) \quad \text{[alternating canonicalize: bubble swap toward repr-sorted args]\,(axiom)} \\
\left(\tilde{\iota}_{\eta}\!\left(\tilde{\iota}_{\omega}\!\left(V\right)\right)\right)\!\left(\xi\right) \to \left(\tilde{\iota}_{\omega}\!\left(V\right)\right)\!\left(\eta,\, \xi\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{\iota}_{\omega}\!\left(V\right)\right)\!\left(\eta,\, \xi\right) \to V\!\left(\omega,\, \eta,\, \xi\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
V\!\left(\omega,\, \eta,\, \xi\right) \to -V\!\left(\eta,\, \omega,\, \xi\right) \quad \text{[alternating canonicalize: bubble swap toward repr-sorted args]\,(axiom)} \\
V\!\left(\eta,\, \omega,\, \xi\right) \to -V\!\left(\eta,\, \xi,\, \omega\right) \quad \text{[alternating canonicalize: bubble swap toward repr-sorted args]\,(axiom)} \\
0\!\left(\xi\right) \to 0 \quad \text{[MultiEval zero-head \ensuremath{\to} 0]\,(axiom)} \\
\left(-V\!\left(\eta,\, \xi,\, \omega\right) - \left(-V\!\left(\eta,\, \xi,\, \omega\right)\right)\right) - 0 \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{gather*}
}

In [4]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] MultiEval head linearity [axiom]
    (ι̃_ω(ι̃_η(V)) + ι̃_η(ι̃_ω(V)))(ξ)
 -> (ι̃_ω(ι̃_η(V))(ξ) + ι̃_η(ι̃_ω(V))(ξ))

[2] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_ω(ι̃_η(V))(ξ)
 -> ι̃_η(V)(ω, ξ)

[3] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_η(V)(ω, ξ)
 -> V(η, ω, ξ)

[4] alternating canonicalize: bubble swap toward repr-sorted args [axiom]
    V(η, ω, ξ)
 -> (-V(η, ξ, ω))

[5] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_η(ι̃_ω(V))(ξ)
 -> ι̃_ω(V)(η, ξ)

[6] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_ω(V)(η, ξ)
 -> V(ω, η, ξ)

[7] alternating canonicalize: bubble swap toward repr-sorted args [axiom]
    V(ω, η, ξ)
 -> (-V(η, ω, ξ))

[8] alternating canonicalize: bubble swap toward repr-sorted args [axiom]
    V(η, ω, ξ)
 -> (-V(η, ξ, ω))

[9] MultiEval zero-head → 0 [axiom]
    0(ξ)
 -> 0

[10] simplify 
    (((-V(η, ξ, ω)) + (-(-V(η, ξ, ω)))) + (-0))
 -> 0



## 3. Relation 2: $\tilde{d}^2 = 0$ (Poisson)

$$
\tilde{d}\bigl(\tilde{d} V\bigr) \;=\; 0
\qquad \bigl(\pi \text{ Poisson, i.e. } [\pi,\pi]_{\mathrm{SN}} = 0\bigr).
$$

**Shape.** This relation closes at the *operator* level via the
closure axiom `TildeDSquaredPoissonDefinition` (Aux-5). The axiom is
gated by the Poisson assumption: if `prob.assume_poisson()` is not
called, it remains a no-op in the engine and the proof fails, i.e.
the system can also actually decide "is $\pi$ Poisson?". Since the
axiom matches **before** the multi-eval expansion, any non-empty
`etas` tuple works; we provide $(\eta, \xi)$.


In [5]:
prob, _, _, V, eta, xi = _build(form_names=("η", "ξ"), V_degree=1)
prob.assume_poisson()  # opens [π,π]_SN = 0  -> Aux-5 active

lhs = Act(
    TildeExteriorDerivative(prob.pi),
    Act(TildeExteriorDerivative(prob.pi), V),
)
chain = prove("(1-VF, eval η,ξ) d̃(d̃ V) = 0  [π Poisson]",
              prob, lhs, Integer(0), etas=(eta, xi))
display_chain(chain)


(1-VF, eval η,ξ) d̃(d̃ V) = 0  [π Poisson] -> closed in 4 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\tilde{d}\!\left(\tilde{d}\!\left(V\right)\right) \to 0 \quad \text{[\ensuremath{\tilde{d}}² V = 0  (\ensuremath{\pi} Poisson) [\ensuremath{\pi}]]\,(axiom)} \\
0\!\left(\eta,\, \xi\right) \to 0 \quad \text{[MultiEval zero-head \ensuremath{\to} 0]\,(axiom)} \\
0\!\left(\eta,\, \xi\right) \to 0 \quad \text{[MultiEval zero-head \ensuremath{\to} 0]\,(axiom)} \\
0 - 0 \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{gather*}
}

In [6]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] d̃² V = 0  (π Poisson) [π] [axiom]
    d̃_π(d̃_π(V))
 -> 0

[2] MultiEval zero-head → 0 [axiom]
    0(η, ξ)
 -> 0

[3] MultiEval zero-head → 0 [axiom]
    0(η, ξ)
 -> 0

[4] simplify 
    (0 + (-0))
 -> 0



## 4. Relation 3: Cartan's magic formula

$$
\tilde{\mathcal{L}}_\omega V \;=\; \tilde{d}\bigl(\tilde{\iota}_\omega V\bigr)
   \;+\; \tilde{\iota}_\omega\bigl(\tilde{d} V\bigr).
$$

**Shape.** On the tilde side this is the *defining* equation of
$\tilde{\mathcal{L}}$ (added to the engine in Phase 14.F as
`TildeCartanMagicDefinition`). When evaluated on a 1-vector $V$ and a
single 1-form $\eta$, both sides reduce to a function; the intrinsic
rules + the Aux-6 bridge convert a bare $\tilde{\iota}_\omega V$
into the scalar `MultiEval(V, ω)`.


In [7]:
prob, _, _, V, omega, eta = _build(V_degree=1)

lhs = Act(TildeLieDerivative(omega, prob.pi), V)
rhs = Sum.make(
    Act(TildeExteriorDerivative(prob.pi),
        Act(TildeInteriorProduct(omega), V)),
    Act(TildeInteriorProduct(omega),
        Act(TildeExteriorDerivative(prob.pi), V)),
)
chain = prove("(1-VF, eval η) L̃_ω V = d̃(ι̃_ω V) + ι̃_ω(d̃ V)",
              prob, lhs, rhs, etas=(eta,))
display_chain(chain)


(1-VF, eval η) L̃_ω V = d̃(ι̃_ω V) + ι̃_ω(d̃ V) -> closed in 11 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(\tilde{\mathcal{L}}_{\omega}\!\left(V\right)\right)\!\left(\eta\right) \to \left(\pi\sharp\!\left(\omega\right)\right)\!\left(V\!\left(\eta\right)\right) - V\!\left(\left[\omega,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (\ensuremath{\tilde{L}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, \ensuremath{\dots}) \ensuremath{-} \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, \ensuremath{\dots}, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left[\omega,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\langle \pi\sharp\!\left(\omega\right),\, \eta \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
V\!\left(L_\pi\sharp(\omega)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\omega\right) - d\!\left(\langle \pi\sharp\!\left(\omega\right),\, \eta \rangle\right)\right) \to V\!\left(L_\pi\sharp(\omega)\!\left(\eta\right)\right) - V\!\left(L_\pi\sharp(\eta)\!\left(\omega\right)\right) - V\!\left(d\!\left(\langle \pi\sharp\!\left(\omega\right),\, \eta \rangle\right)\right) \quad \text{[MultiEval arg slot linearity]\,(axiom)} \\
\tilde{d}\!\left(\tilde{\iota}_{\omega}\!\left(V\right)\right) \to \tilde{d}\!\left(V\!\left(\omega\right)\right) \quad \text{[bare \ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega}(V) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(V, \ensuremath{\omega}))  (V deg 1)]\,(axiom)} \\
\left(\tilde{d}\!\left(V\!\left(\omega\right)\right) + \tilde{\iota}_{\omega}\!\left(\tilde{d}\!\left(V\right)\right)\right)\!\left(\eta\right) \to \left(\tilde{d}\!\left(V\!\left(\omega\right)\right)\right)\!\left(\eta\right) + \left(\tilde{\iota}_{\omega}\!\left(\tilde{d}\!\left(V\right)\right)\right)\!\left(\eta\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(\tilde{d}\!\left(V\!\left(\omega\right)\right)\right)\!\left(\eta\right) \to \left(\pi\sharp\!\left(\eta\right)\right)\!\left(V\!\left(\omega\right)\right) \quad \text{[\ensuremath{\tilde{d}} intrinsic (Koszul) [\ensuremath{\pi}]: (\ensuremath{\tilde{d}}V)(\ensuremath{\eta}\_0, \ensuremath{\dots}) = \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(\ensuremath{\dots}) + \ensuremath{\Sigma} \ensuremath{\pm}V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{\iota}_{\omega}\!\left(\tilde{d}\!\left(V\right)\right)\right)\!\left(\eta\right) \to \left(\tilde{d}\!\left(V\right)\right)\!\left(\omega,\, \eta\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{d}\!\left(V\right)\right)\!\left(\omega,\, \eta\right) \to \left(\pi\sharp\!\left(\omega\right)\right)\!\left(V\!\left(\eta\right)\right) - \left(\pi\sharp\!\left(\eta\right)\right)\!\left(V\!\left(\omega\right)\right) - V\!\left(\left[\omega,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{d}} intrinsic (Koszul) [\ensuremath{\pi}]: (\ensuremath{\tilde{d}}V)(\ensuremath{\eta}\_0, \ensuremath{\dots}) = \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(\ens

In [8]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·V(η_1, …) − Σ V(η_1, …, [ω, η_i]_K, …) [axiom]
    L̃_ω(V)(η)
 -> (π♯(ω)(V(η)) + (-V([·,·]_K[π](ω, η))))

[2] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](ω, η)
 -> (L_π♯(ω)(η) + (-L_π♯(η)(ω)) + (-d(⟨π♯(ω), η⟩)))

[3] MultiEval arg slot linearity [axiom]
    V((L_π♯(ω)(η) + (-L_π♯(η)(ω)) + (-d(⟨π♯(ω), η⟩))))
 -> (V(L_π♯(ω)(η)) + (-V(L_π♯(η)(ω))) + (-V(d(⟨π♯(ω), η⟩))))

[4] bare ι̃_ω(V) inside Act(D, _) → Act(D, MultiEval(V, ω))  (V deg 1) [axiom]
    d̃_π(ι̃_ω(V))
 -> d̃_π(V(ω))

[5] MultiEval head linearity [axiom]
    (d̃_π(V(ω)) + ι̃_ω(d̃_π(V)))(η)
 -> (d̃_π(V(ω))(η) + ι̃_ω(d̃_π(V))(η))

[6] d̃ intrinsic (Koszul) [π]: (d̃V)(η_0, …) = Σ ±π^♯(η_i)·V(…) + Σ ±V([η_i,η_j]_K, …) [axiom]
    d̃_π(V(ω))(η)
 -> π♯(η)(V(ω))

[7] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_ω(d̃_π(V))(η)
 -> d̃_π(V)(ω, η)

[8] d̃ intrinsic (Koszul) [π]: (d̃V)(η_0, …) = Σ ±π^♯(η_i)·V(…) + Σ ±V([η_i,η_j]_K, …) [axi

## 5. Relation 4: $[\tilde{\mathcal{L}}_\alpha, \tilde{\iota}_\beta] = \tilde{\iota}_{[\alpha,\beta]_K}$

$$
\tilde{\mathcal{L}}_\alpha\bigl(\tilde{\iota}_\beta V\bigr)
   \;-\; \tilde{\iota}_\beta\bigl(\tilde{\mathcal{L}}_\alpha V\bigr)
   \;=\; \tilde{\iota}_{[\alpha,\beta]_K}\,V.
$$

**Shape.** Pick $V$ as a 2-vector; $\tilde{\iota}_\beta V$ is a
1-vector that drops to a function on a single 1-form $\eta$. The
right-side Koszul bracket $[\alpha,\beta]_K$ is opened by
`KoszulProblem`'s registered bracket-expansion rule into the
Lichnerowicz Cartan form
$\mathcal{L}_{\rho\alpha}\beta - \mathcal{L}_{\rho\beta}\alpha -
d\langle\rho\alpha,\beta\rangle$. Then $\tilde{\iota}$-linearity
distributes term by term and the engine completes the closure.


In [9]:
prob, _, _, V, alpha, beta = _build(form_names=("α", "β"), V_degree=2)
eta = Symbol("η")
prob.registry.declare(eta, Graded(degree=1))

lhs = Sum.make(
    Act(TildeLieDerivative(alpha, prob.pi),
        Act(TildeInteriorProduct(beta), V)),
    Neg(Act(TildeInteriorProduct(beta),
            Act(TildeLieDerivative(alpha, prob.pi), V))),
)
bracket_form = prob.bracket(alpha, beta)
rhs = Act(TildeInteriorProduct(bracket_form), V)
chain = prove("(2-VF, eval η) [L̃_α, ι̃_β] V = ι̃_[α,β]_K V",
              prob, lhs, rhs, etas=(eta,))
display_chain(chain)


(2-VF, eval η) [L̃_α, ι̃_β] V = ι̃_[α,β]_K V -> closed in 22 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right) - \tilde{\iota}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta\right) \to \left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) - \left(\tilde{\iota}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) \to \left(\pi\sharp\!\left(\alpha\right)\right)\!\left(\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\eta\right)\right) - \left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (\ensuremath{\tilde{L}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, \ensuremath{\dots}) \ensuremath{-} \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, \ensuremath{\dots}, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\eta\right) \to V\!\left(\beta,\, \eta\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right)\right) \to V\!\left(\beta,\, L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right)\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
V\!\left(\beta,\, L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right)\right) \to V\!\left(\beta,\, L_\pi\sharp(\alpha)\!\left(\eta\right)\right) - V\!\left(\beta,\, L_\pi\sharp(\eta)\!\left(\alpha\right)\right) - V\!\left(\beta,\, d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right)\right) \quad \text{[MultiEval arg slot linearity]\,(axiom)} \\
V\!\left(\beta,\, L_\pi\sharp(\alpha)\!\left(\eta\right)\right) \to -V\!\left(L_\pi\sharp(\alpha)\!\left(\eta\right),\, \beta\right) \quad \text{[alternating canonicalize: bubble swap toward repr-sorted args]\,(axiom)} \\
V\!\left(\beta,\, L_\pi\sharp(\eta)\!\left(\alpha\right)\right) \to -V\!\left(L_\pi\sharp(\eta)\!\left(\alpha\right),\, \beta\right) \quad \text{[alternating canonicalize: bubble swap toward repr-sorted args]\,(axiom)} \\
V\!\left(\beta,\, d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right)\right) \

In [10]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] MultiEval head linearity [axiom]
    (L̃_α(ι̃_β(V)) + (-ι̃_β(L̃_α(V))))(η)
 -> (L̃_α(ι̃_β(V))(η) + (-ι̃_β(L̃_α(V))(η)))

[2] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·V(η_1, …) − Σ V(η_1, …, [ω, η_i]_K, …) [axiom]
    L̃_α(ι̃_β(V))(η)
 -> (π♯(α)(ι̃_β(V)(η)) + (-ι̃_β(V)([·,·]_K[π](α, η))))

[3] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_β(V)(η)
 -> V(β, η)

[4] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](α, η)
 -> (L_π♯(α)(η) + (-L_π♯(η)(α)) + (-d(⟨π♯(α), η⟩)))

[5] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_β(V)((L_π♯(α)(η) + (-L_π♯(η)(α)) + (-d(⟨π♯(α), η⟩))))
 -> V(β, (L_π♯(α)(η) + (-L_π♯(η)(α)) + (-d(⟨π♯(α), η⟩))))

[6] MultiEval arg slot linearity [axiom]
    V(β, (L_π♯(α)(η) + (-L_π♯(η)(α)) + (-d(⟨π♯(α), η⟩))))
 -> (V(β, L_π♯(α)(η)) + (-V(β, L_π♯(η)(α))) + (-V(β, d(⟨π♯(α), η⟩))))

[7] alternating canonicalize: bubble swap toward repr-sorted args [axiom]
    V(β, L_π♯(α)(η))
 -> (-V(L_π♯(α)(η), β))

[8] 

## 6. Relation 5: $[\tilde{\mathcal{L}}_\alpha, \tilde{d}] = 0$ (Poisson)

$$
\tilde{\mathcal{L}}_\alpha\bigl(\tilde{d} V\bigr)
   \;-\; \tilde{d}\bigl(\tilde{\mathcal{L}}_\alpha V\bigr)
   \;=\; 0
\qquad \bigl([\pi,\pi]_{\mathrm{SN}} = 0\bigr).
$$

**Shape (the 14.G closure).** Evaluated on a 1-vector $V$ + two
1-forms $(\eta,\xi)$, the difference initially blows up to ~24
SN-Jacobi terms. Engine pipeline:

1. **Slot-Lie commutator** + **anchor Lie homomorphism** + **pairing
   Leibniz** rules drop the residue to 9 terms.
2. `WrappedPairingAnchorAntisymmetryDefinition` catches 2 paired
   cancellations of the form
   $\langle \pi^\sharp a, b\rangle +
   \langle \pi^\sharp b, a\rangle = 0$ under a common
   `Act`/`MultiEval` wrapper -> 5 terms.
3. `TildeSnJacobiResidueDefinition` recognizes the remaining 5-term
   shape as the canonical form of the Schouten–Nijenhuis Jacobi
   obstruction and zeros it under Poisson.

`prob.assume_poisson()` activates both rules.


In [11]:
prob, _, _, V, alpha, eta = _build(form_names=("α", "η"), V_degree=1)
prob.assume_poisson()
xi = Symbol("ξ")
prob.registry.declare(xi, Graded(degree=1))

lhs = Sum.make(
    Act(TildeLieDerivative(alpha, prob.pi),
        Act(TildeExteriorDerivative(prob.pi), V)),
    Neg(Act(TildeExteriorDerivative(prob.pi),
            Act(TildeLieDerivative(alpha, prob.pi), V))),
)
chain = prove("(1-VF, eval η,ξ) [L̃_α, d̃] V = 0  [π Poisson]",
              prob, lhs, Integer(0), etas=(eta, xi))
display_chain(chain)


(1-VF, eval η,ξ) [L̃_α, d̃] V = 0  [π Poisson] -> closed in 133 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(V\right)\right) - \tilde{d}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) \to \left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) \to \left(\pi\sharp\!\left(\alpha\right)\right)\!\left(\left(\tilde{d}\!\left(V\right)\right)\!\left(\eta,\, \xi\right)\right) - \left(\tilde{d}\!\left(V\right)\right)\!\left(\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]},\, \xi\right) - \left(\tilde{d}\!\left(V\right)\right)\!\left(\eta,\, \left[\alpha,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (\ensuremath{\tilde{L}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, \ensuremath{\dots}) \ensuremath{-} \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, \ensuremath{\dots}, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{d}\!\left(V\right)\right)\!\left(\eta,\, \xi\right) \to \left(\pi\sharp\!\left(\eta\right)\right)\!\left(V\!\left(\xi\right)\right) - \left(\pi\sharp\!\left(\xi\right)\right)\!\left(V\!\left(\eta\right)\right) - V\!\left(\left[\eta,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{d}} intrinsic (Koszul) [\ensuremath{\pi}]: (\ensuremath{\tilde{d}}V)(\ensuremath{\eta}\_0, \ensuremath{\dots}) = \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(\ensuremath{\dots}) + \ensuremath{\Sigma} \ensuremath{\pm}V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left[\eta,\, \xi\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\eta)\!\left(\xi\right) - L_\pi\sharp(\xi)\!\left(\eta\right) - d\!\left(\langle \pi\sharp\!\left(\eta\right),\, \xi \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
V\!\left(L_\pi\sharp(\eta)\!\left(\xi\right) - L_\pi\sharp(\xi)\!\left(\eta\right) - d\!\left(\langle \pi\sharp\!\left(\eta\right),\, \xi \rangle\right)\right) \to V\!\left(L_\pi\sharp(\eta)\!\left(\xi\right)\right) - V\!\left(L_\pi\sharp(\xi)\!\left(\eta\right)\right) - V\!\left(d\!\left(\langle \pi\sharp\!\left(\eta\right),\, \xi \rangle\right)\right) \quad \text{[MultiEval arg slot linearity]\,(axiom)} \\
\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
\left(\tilde{d}\!\left(V\right)\right)\!\left(L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right),\, \xi\right) \to \left(\pi\sharp\!\left(L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\le

In [12]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] MultiEval head linearity [axiom]
    (L̃_α(d̃_π(V)) + (-d̃_π(L̃_α(V))))(η, ξ)
 -> (L̃_α(d̃_π(V))(η, ξ) + (-d̃_π(L̃_α(V))(η, ξ)))

[2] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·V(η_1, …) − Σ V(η_1, …, [ω, η_i]_K, …) [axiom]
    L̃_α(d̃_π(V))(η, ξ)
 -> (π♯(α)(d̃_π(V)(η, ξ)) + (-d̃_π(V)([·,·]_K[π](α, η), ξ)) + (-d̃_π(V)(η, [·,·]_K[π](α, ξ))))

[3] d̃ intrinsic (Koszul) [π]: (d̃V)(η_0, …) = Σ ±π^♯(η_i)·V(…) + Σ ±V([η_i,η_j]_K, …) [axiom]
    d̃_π(V)(η, ξ)
 -> (π♯(η)(V(ξ)) + (-π♯(ξ)(V(η))) + (-V([·,·]_K[π](η, ξ))))

[4] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](η, ξ)
 -> (L_π♯(η)(ξ) + (-L_π♯(ξ)(η)) + (-d(⟨π♯(η), ξ⟩)))

[5] MultiEval arg slot linearity [axiom]
    V((L_π♯(η)(ξ) + (-L_π♯(ξ)(η)) + (-d(⟨π♯(η), ξ⟩))))
 -> (V(L_π♯(η)(ξ)) + (-V(L_π♯(ξ)(η))) + (-V(d(⟨π♯(η), ξ⟩))))

[6] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](α, η)
 -> (L_π♯(α)(η) + (-L_π♯(η)(α)) + (-d(⟨π♯(α), η⟩)))

[7] d̃ intrinsic (Koszul) [π]: (d̃V)(η_

## 7. Relation 6: $[\tilde{\mathcal{L}}_\alpha, \tilde{\mathcal{L}}_\beta] = \tilde{\mathcal{L}}_{[\alpha,\beta]_K}$ (Poisson)

$$
\tilde{\mathcal{L}}_\alpha\bigl(\tilde{\mathcal{L}}_\beta V\bigr)
   \;-\; \tilde{\mathcal{L}}_\beta\bigl(\tilde{\mathcal{L}}_\alpha V\bigr)
   \;=\; \tilde{\mathcal{L}}_{[\alpha,\beta]_K}\,V
\qquad \bigl([\pi,\pi]_{\mathrm{SN}} = 0\bigr).
$$

**Shape (the 14.G closure).** Same pipeline as relation 5: the
slot-Lie / anchor-Lie / pairing-Leibniz expansions leave a 9-term
residue, `WrappedPairingAnchorAntisymmetryDefinition` reduces it to 5
via 2 paired cancellations, and `TildeSnJacobiResidueDefinition`
closes the SN-Jacobi obstruction. The expression drops to a function
on a 1-vector $V$ and a single 1-form $\eta$.


In [13]:
prob, _, _, V, alpha, beta = _build(form_names=("α", "β"), V_degree=1)
prob.assume_poisson()
eta = Symbol("η")
prob.registry.declare(eta, Graded(degree=1))

lhs = Sum.make(
    Act(TildeLieDerivative(alpha, prob.pi),
        Act(TildeLieDerivative(beta, prob.pi), V)),
    Neg(Act(TildeLieDerivative(beta, prob.pi),
            Act(TildeLieDerivative(alpha, prob.pi), V))),
)
bracket_form = prob.bracket(alpha, beta)
rhs = Act(TildeLieDerivative(bracket_form, prob.pi), V)
chain = prove("(1-VF, eval η) [L̃_α, L̃_β] V = L̃_[α,β]_K V  [π Poisson]",
              prob, lhs, rhs, etas=(eta,))
display_chain(chain)


(1-VF, eval η) [L̃_α, L̃_β] V = L̃_[α,β]_K V  [π Poisson] -> closed in 119 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right) - \tilde{\mathcal{L}}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta\right) \to \left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) - \left(\tilde{\mathcal{L}}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) \to \left(\pi\sharp\!\left(\alpha\right)\right)\!\left(\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\eta\right)\right) - \left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (\ensuremath{\tilde{L}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, \ensuremath{\dots}) \ensuremath{-} \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, \ensuremath{\dots}, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\eta\right) \to \left(\pi\sharp\!\left(\beta\right)\right)\!\left(V\!\left(\eta\right)\right) - V\!\left(\left[\beta,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (\ensuremath{\tilde{L}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, \ensuremath{\dots}) \ensuremath{-} \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, \ensuremath{\dots}, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left[\beta,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\beta)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\beta\right) - d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
V\!\left(L_\pi\sharp(\beta)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\beta\right) - d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right)\right) \to V\!\left(L_\pi\sharp(\beta)\!\left(\eta\right)\right) - V\!\left(L_\pi\sharp(\eta)\!\left(\beta\right)\right) - V\!\left(d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right)\right) \quad \text{[MultiEval arg slot linearity]\,(axiom)} \\
\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right)\right) \to \left(\pi\sharp\!\left(\beta\right)\right)\!\left(V\!\left(L_\pi\sharp(\alpha)\!\left(\et

In [14]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] MultiEval head linearity [axiom]
    (L̃_α(L̃_β(V)) + (-L̃_β(L̃_α(V))))(η)
 -> (L̃_α(L̃_β(V))(η) + (-L̃_β(L̃_α(V))(η)))

[2] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·V(η_1, …) − Σ V(η_1, …, [ω, η_i]_K, …) [axiom]
    L̃_α(L̃_β(V))(η)
 -> (π♯(α)(L̃_β(V)(η)) + (-L̃_β(V)([·,·]_K[π](α, η))))

[3] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·V(η_1, …) − Σ V(η_1, …, [ω, η_i]_K, …) [axiom]
    L̃_β(V)(η)
 -> (π♯(β)(V(η)) + (-V([·,·]_K[π](β, η))))

[4] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](β, η)
 -> (L_π♯(β)(η) + (-L_π♯(η)(β)) + (-d(⟨π♯(β), η⟩)))

[5] MultiEval arg slot linearity [axiom]
    V((L_π♯(β)(η) + (-L_π♯(η)(β)) + (-d(⟨π♯(β), η⟩))))
 -> (V(L_π♯(β)(η)) + (-V(L_π♯(η)(β))) + (-V(d(⟨π♯(β), η⟩))))

[6] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](α, η)
 -> (L_π♯(α)(η) + (-L_π♯(η)(α)) + (-d(⟨π♯(α), η⟩)))

[7] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·V(η_1, …) − Σ V(η_1, …, [ω, η_i]_K, …) [axiom]
    L̃_β(V)(

## Conclusion

All six tilde Cartan relations close syntactically on a generic
multivector $V$ via a single `prob.prove_tilde_cartan` call.

| # | Relation | $V$ degree | Eval | Poisson? | Closure channel |
|---|---|---|---|---|---|
| 1 | $\tilde{\iota}\tilde{\iota}$ anti-commute | 2 | $\xi$ | no | MultiEval alternating |
| 2 | $\tilde{d}^2 V = 0$ | 1 | $\eta,\xi$ | **yes** | `TildeDSquaredPoisson` (Aux-5) |
| 3 | Cartan magic | 1 | $\eta$ | no | `TildeCartanMagic` (14.F) |
| 4 | $[\tilde{\mathcal{L}}, \tilde{\iota}]$ commutator | 2 | $\eta$ | no | bracket-expansion + intrinsic |
| 5 | $[\tilde{\mathcal{L}}, \tilde{d}] = 0$ | 1 | $\eta,\xi$ | **yes** | 14.G: WrappedPairingAntisym + SN-Jacobi recognizer |
| 6 | $[\tilde{\mathcal{L}}, \tilde{\mathcal{L}}]$ commutator | 1 | $\eta$ | **yes** | 14.G: same pipeline |

All proofs run on the same 28-rule engine; the `Definition` from
which each step originated is recorded in `ProofStep.rule`, and
`display_chain` renders it as readable LaTeX.